# **Scraping Planos Nio Telecom**

Página do provedor: https://www.niointernet.com.br/

In [1]:
# Bibliotecas
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from datetime import date
import re

## **1. Extração**

In [2]:
# Obter lista de planos
texto_html = open("../../data/scraped/nio.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
planos = soup.find_all("div", class_="new-home__plans-card-wrapper")
print(f"{len(planos)} cartões de planos capturados.")

4 cartões de planos capturados.


In [6]:
# Extrair informações dos planos
def raspar_plano(plano):
    velocidade = plano.find("p", class_="new-home__plans-card__speed").text
    preco = plano.find("span", class_="new-home__plans-card__price-value").text
    pre_sva = plano.find_all("img")
    lista_sva = [i['src'] for i in pre_sva]
    svas = [i["src"].split("/")[-1] for i in pre_sva]
    svas = ", ".join(svas)

    return {"velocidade": velocidade,
            "preco": preco,
            "svas": svas}

dados = []

for i in planos:
    dados_plano = raspar_plano(i)
    dados.append(dados_plano)

df = pd.DataFrame(dados)
df

,velocidade,preco,svas
0,600 Mega,110,"rastros.svg, icon-wifi-dark.svg, icon-wifi-spe..."
1,800 Mega,140,"rastros.svg, icon-wifi-dark.svg, icon-smartpho..."
2,1 Giga,170,"rastros.svg, icon-wifi-light.svg, icon-smartph..."
3,1 Giga,200,"rastros.svg, icon-wifi-plus-light.svg, icon-sm..."


## **2. Tratamento**

In [ ]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "Nio"

# Velocidade
vel = df_tratado["velocidade"].str.lower()
num = vel.str.extract(r"(\d+)", expand=False).astype(float)
df_tratado["VELOCIDADE"] = np.where(
    vel.str.contains(r"giga|gb", na=False),
    num * 1000,
    num
)

# PRECO
df_tratado["PRECO"] = (
    df_tratado["preco"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)

# SVAs
df_tratado["SVAs"] = df_tratado["svas"]

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

## **3. Carga**

In [ ]:
# Salvando
provedor = "NIO.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")